# nb_eventlog_analysis — paste a URL, get a real diagnosis
Downloads a Spark **event log** from Fabric and analyses it. The event log is the raw stream the
Spark UI is rendered from — every task, stage, accumulator and executor lifecycle event — so it
supports analysis the UI does not offer.

**Nothing here is hardcoded.** Every number is derived from the actual event stream
(`SparkListenerTaskEnd`, `SparkListenerStageCompleted`, `SparkListenerEnvironmentUpdate`, …).
Point it at a different application and it reports that application.

**What is proven here:** the parser and all analysis run against a **real Spark event log generated
live in this session**, including a deliberately skewed join, so the skew detector demonstrably
fires on real data rather than a fixture. The Fabric *download* call is the one part needing a live
session; its URL handling is fully tested below, the HTTP call itself is documented not exercised.

In [1]:
NOTEBOOK_NAME = "nb_eventlog_analysis"
FABRIC_URL    = ""      # paste ANY Fabric URL shape - see section 1
EVENTLOG_PATH = ""      # or a downloaded .zip / .gz / directory
DEMO_MODE     = True
SKEW_RATIO    = 3.0
SPILL_MB      = 64

In [2]:
import sys, os, glob, shutil
for _p in ("/lakehouse/default/Files/code", os.getcwd()):
    if os.path.isdir(_p) and _p not in sys.path:
        sys.path.append(_p)
from spark_eventlog_analyzer import (
    from_file, from_fabric_url, parse_fabric_url, UrlProblem, SparkAppRef)
print("analyzer loaded")

analyzer loaded


## 1 — The URL problem, solved
"Bad URL" is almost always one of five things. Four are handled silently; the fifth gets an
explanation instead of an opaque 400.

In [3]:
WS="aaaabbbb-0000-cccc-1111-dddd2222eeee"; IT="11bb11bb-cc22-dd33-ee44-55ff55ff55ff"
LV="0a0a0a0a-1111-bbbb-2222-3c3c3c3c3c3c"; AP="application_1741176604085_0001"
API="https://api.fabric.microsoft.com/v1"
samples = [
 ("Full REST URL",               f"{API}/workspaces/{WS}/notebooks/{IT}/livySessions/{LV}/applications/{AP}/1/logs"),
 ("Missing /attemptId/",         f"{API}/workspaces/{WS}/notebooks/{IT}/livySessions/{LV}/applications/{AP}"),
 ("Relative path (no host)",     f"v1/workspaces/{WS}/notebooks/{IT}/livySessions/{LV}/applications/{AP}/1/logs"),
 ("Quotes + query + fragment",   f'"{API}/workspaces/{WS}/notebooks/{IT}/livySessions/{LV}/applications/{AP}/1/logs?x=1#f/"'),
 ("Spark Job Definition, try 2", f"{API}/workspaces/{WS}/sparkJobDefinitions/{IT}/livySessions/{LV}/applications/{AP}/2/logs"),
 ("Portal URL (ids in query)",   f"https://app.fabric.microsoft.com/groups/{WS}/synapsenotebooks/{IT}?sessionId={LV}&appId={AP}"),
]
for label, u in samples:
    r = parse_fabric_url(u)
    print(f"OK   {label:<30} -> kind={r.item_kind:<20} attempt={r.attempt_id}")
    assert r.workspace_id == WS and r.livy_id == LV and r.app_id == AP
print()
for label, u in [("Empty", ""), ("No ids at all", "https://app.fabric.microsoft.com/sparkui/history/jobs/")]:
    try:
        parse_fabric_url(u); print(f"FAIL {label}")
    except UrlProblem as e:
        print(f"OK   rejected {label!r}:")
        for ln in str(e).splitlines()[:2]: print("       " + ln)

OK   Full REST URL                  -> kind=notebooks            attempt=1
OK   Missing /attemptId/            -> kind=notebooks            attempt=1
OK   Relative path (no host)        -> kind=notebooks            attempt=1
OK   Quotes + query + fragment      -> kind=notebooks            attempt=1
OK   Spark Job Definition, try 2    -> kind=sparkJobDefinitions  attempt=2
OK   Portal URL (ids in query)      -> kind=notebooks            attempt=1

OK   rejected 'Empty':
       Empty URL. Paste either the Monitoring-hub application URL or the REST API URL, or use from_ids() with the ids from Recent runs.
OK   rejected 'No ids at all':
       Could not parse a Spark application reference from that URL.
         Problems: expected 3 GUIDs (workspace, item, livySession) but found 0; no application id of the form application_<epoch>_<seq> found.


In [4]:
ref = SparkAppRef(WS, "notebooks", IT, LV, AP, 1)
print("event log download endpoint:")
print("  " + ref.logs_url())
print("\nsame ids, other History-Server-compatible endpoints:")
for ep in ("jobs", "stages", "executors", "environment"):
    print(f"  {ep:<12} .../applications/" + ref.api_url(ep).split("/applications/")[1])

event log download endpoint:
  https://api.fabric.microsoft.com/v1/workspaces/aaaabbbb-0000-cccc-1111-dddd2222eeee/notebooks/11bb11bb-cc22-dd33-ee44-55ff55ff55ff/livySessions/0a0a0a0a-1111-bbbb-2222-3c3c3c3c3c3c/applications/application_1741176604085_0001/1/logs

same ids, other History-Server-compatible endpoints:
  jobs         .../applications/application_1741176604085_0001/1/jobs
  stages       .../applications/application_1741176604085_0001/1/stages
  executors    .../applications/application_1741176604085_0001/1/executors
  environment  .../applications/application_1741176604085_0001/1/environment


## 2 — Produce a REAL event log to analyse
`spark.eventLog.enabled` writes the same JSON-lines format Fabric produces. The workload contains a
**deliberately skewed join** (70% of rows on one key) so the detector has something genuine to find.

In [5]:
from pyspark.sql import SparkSession, functions as F
EVDIR = "/tmp/nb_evlogs"
if DEMO_MODE and not EVENTLOG_PATH and not FABRIC_URL:
    shutil.rmtree(EVDIR, ignore_errors=True); os.makedirs(EVDIR)
    try: spark.stop()
    except Exception: pass
    from delta import configure_spark_with_delta_pip
    _b = (SparkSession.builder.appName("eventlog-demo").master("local[4]")
          .config("spark.eventLog.enabled","true").config("spark.eventLog.dir", f"file://{EVDIR}")
          .config("spark.driver.memory","1500m")
          .config("spark.sql.adaptive.enabled","false")
          .config("spark.sql.shuffle.partitions","16")
          .config("spark.sql.autoBroadcastJoinThreshold","-1")
          .config("spark.sql.extensions","io.delta.sql.DeltaSparkSessionExtension")
          .config("spark.sql.catalog.spark_catalog","org.apache.spark.sql.delta.catalog.DeltaCatalog"))
    spark = configure_spark_with_delta_pip(_b).getOrCreate()
    spark.sparkContext.setLogLevel("ERROR")
    left = (spark.range(0, 1_200_000)
            .withColumn("k", F.when(F.col("id") % 10 < 7, F.lit(1)).otherwise(F.col("id") % 500))
            .withColumn("pad", F.lit("A"*150)))
    right = spark.range(0, 500).withColumnRenamed("id","k").withColumn("rpad", F.lit("B"*150))
    left.join(right, "k").groupBy("k").agg(F.count("*").alias("n"), F.max("pad")).collect()
    spark.stop()
    logs = glob.glob(f"{EVDIR}/*")
    print("event log written:", os.path.basename(logs[0]), f"({os.path.getsize(logs[0])//1024} KB)")

26/08/17 10:12:34 WARN Utils: Your hostname, vm resolves to a loopback address: 127.0.0.1; using 192.0.2.2 instead (on interface eth0)
26/08/17 10:12:34 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/usr/local/lib/python3.12/dist-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-9dbac31e-e6ad-4648-834d-4f6f1fb46f68;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.0 in central


	found io.delta#delta-storage;3.2.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 323ms :: artifacts dl 23ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.2.0 from central in [default]
	io.delta#delta-storage;3.2.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   |   0   |   0   |   0   ||   3   |   0   |
	---------------------------------------------------------------------
:: retrieving :: org.apache.spark#spark-submit-parent-9dbac31e-e6ad-4648-834d-4f6f1fb46f68
	confs: [default]
	0 artifacts copied, 3 already retrieved (0kB/16ms)


26/08/17 10:12:35 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


event log written: local-1786961557772 (320 KB)


## 3 — Analyse (same call, whatever the source)

In [6]:
if FABRIC_URL:      analysis = from_fabric_url(FABRIC_URL)
elif EVENTLOG_PATH: analysis = from_file(EVENTLOG_PATH)
else:               analysis = from_file(glob.glob(f"{EVDIR}/*")[0])
print(analysis.report())

EVENT LOG ANALYSIS - eventlog-demo (local-1786961557772)
duration=15.74s  jobs=1  stages=3  tasks=24  executors=1
shuffle read=0.2MB  write=0.2MB  spill=0.0MB  input=0.0MB
config: AQE=false  NEE=None  ANSI=None  shuffle.partitions=16

SLOWEST STAGES (critical path)
  stage 2.0      3.8s (64.0%)  tasks=16    skew= 3.8x  spill=     0MB  collect at /tmp/ipykernel_695/867481039.py:2
  stage 1.0      1.3s (21.0%)  tasks=4     skew= 1.0x  spill=     0MB  collect at /tmp/ipykernel_695/867481039.py:2
  stage 0.0      0.9s (15.1%)  tasks=4     skew= 1.0x  spill=     0MB  collect at /tmp/ipykernel_695/867481039.py:2

FINDINGS (1 critical, 0 warn)
  [E-SKEW/critical] stage 2.0: Task duration skew 3.8x (max 1733ms vs median 457ms)
      fix: One task dominates the stage. If it is a join, confirm AQE skew-join is on; for aggregation skew, salt the key. Check whether the hot key is NULL.


In [7]:
s = analysis.summary()
print("summary keys:", list(s))
print(f"\nconfig read FROM THE LOG: AQE={s['aqe_enabled']}  shuffle.partitions={s['shuffle_partitions']}")
assert s["shuffle_partitions"] == "16", "config must come from the event log, not be assumed"
assert s["jobs"] >= 1 and s["tasks"] > 0
print("PASS: configuration derived from the event stream, not hardcoded")

summary keys: ['app_name', 'app_id', 'duration_s', 'jobs', 'stages', 'tasks', 'total_shuffle_read_mb', 'total_shuffle_write_mb', 'total_spill_mb', 'total_input_mb', 'executors', 'sql_executions', 'aqe_enabled', 'nee_enabled', 'ansi_enabled', 'shuffle_partitions']

config read FROM THE LOG: AQE=false  shuffle.partitions=16
PASS: configuration derived from the event stream, not hardcoded


In [8]:
print(f"{'stage':<10}{'wall_s':>8}{'tasks':>7}{'skew':>7}{'spill_MB':>10}{'shuffle_rd_MB':>15}")
for st in analysis.critical_path():
    print(f"{str(st['stage_id'])+'.'+str(st['attempt']):<10}{st['wall_s']:>8.1f}{st['task_count']:>7}"
          f"{st['skew_ratio']:>7.1f}{st['spill_mb']:>10.0f}{st['shuffle_read_mb']:>15.1f}")
skewed = [x for x in analysis.stage_table() if x["skew_ratio"] >= SKEW_RATIO]
print(f"\nstages exceeding skew ratio {SKEW_RATIO}: {[x['stage_id'] for x in skewed]}")
assert skewed, "the deliberately skewed join should have been detected"
print(f"PASS: skew detected on real data - {skewed[0]['skew_ratio']:.1f}x on stage {skewed[0]['stage_id']}")

stage       wall_s  tasks   skew  spill_MB  shuffle_rd_MB
2.0            3.8     16    3.8         0            0.2
1.0            1.3      4    1.0         0            0.0
0.0            0.9      4    1.0         0            0.0

stages exceeding skew ratio 3.0: [2]
PASS: skew detected on real data - 3.8x on stage 2


In [9]:
for f in analysis.findings(skew_ratio=SKEW_RATIO, spill_mb=SPILL_MB):
    print(f"[{f.code}/{f.severity}] {f.scope}: {f.message}")
    print(f"    evidence: {f.evidence}")

[E-SKEW/critical] stage 2.0: Task duration skew 3.8x (max 1733ms vs median 457ms)
    evidence: {'max_ms': 1733, 'median_ms': 457.0, 'tasks': 16}


## 4 — Wiring it into operations
```python
# Scheduled QA notebook: analyse every run that breached its SLA
for r in slow_runs:                       # from etl_run_log (Sec 33)
    a = from_ids(workspace_id=WS, item_id=IT, livy_id=r.livy_id, app_id=r.spark_app_id)
    write_findings(r.run_id, a.findings())
```
A failing pipeline then arrives with its own root-cause analysis attached.

**Cost note:** event logs for a large application can run to hundreds of MB. Download once, cache to
a Lakehouse Files path, analyse from there.

In [10]:
print("Live-mode checklist:")
print("  [ ] FABRIC_URL set (any shape from section 1)")
print("  [ ] inside Fabric (sempy supplies the REST client) OR pass client=")
print("  [ ] read permission on the notebook / SJD / lakehouse item")
print("  [ ] the application has COMPLETED - event logs finalise on completion")

Live-mode checklist:
  [ ] FABRIC_URL set (any shape from section 1)
  [ ] inside Fabric (sempy supplies the REST client) OR pass client=
  [ ] read permission on the notebook / SJD / lakehouse item
  [ ] the application has COMPLETED - event logs finalise on completion
